<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration — Algorithm</b></h1>
</div>


## Purpose and reading order

This notebook is the **algorithm specification** for the corresponding `main.ipynb` in this lab.

It has two layers, designed to be read in either direction:

| Layer | Content | Answers |
| --- | --- | --- |
| **Part I — Complete pseudocode** | One self-contained block with every major branch and failure path explicit | "What is the whole algorithm?" |
| **Part II — Step-by-step elaboration** | Six-section contract for every implementation step | "Why this step, with what inputs, and against what acceptance condition?" |

Names and operations are kept aligned with `main.ipynb`. This notebook specifies the implemented workflow; it does not introduce additional numerical gates that are absent from the executable notebook.

**Standing rule.** The acceptance contract of step *k* is the precondition of step *k+1*.

# Part I — Complete Pseudocode

The single block below is the complete calibration pipeline. It is implementation-neutral but remains faithful to `main.ipynb`: every named quantity is introduced before use, every implemented failure path is explicit, and no extra acceptance threshold is invented.

```text
ALGORITHM Camera_Calibration

  # ─────────────────────────────────────────────────────────────
  # FIXED CALIBRATION SETTINGS
  # ─────────────────────────────────────────────────────────────
  PATTERN          ← (8, 6)
  N_POINTS         ← 48
  SQUARE_SIZE_M    ← 0.03
  MIN_VALID_VIEWS  ← 3
  DATA_DIR         ← "../data/calibration_images"
  OUTPUT_DIR       ← "../outputs/figures"

  # ─────────────────────────────────────────────────────────────
  # 1. CREATE THE SETUP
  # ─────────────────────────────────────────────────────────────
  configure numerical display
  define PATTERN, SQUARE_SIZE_M, MIN_VALID_VIEWS
  define subpixel corner-refinement criteria

  # ─────────────────────────────────────────────────────────────
  # 2. CREATE REPOSITORY-RELATIVE PATHS
  # ─────────────────────────────────────────────────────────────
  IF DATA_DIR does not exist:
      RETURN FAIL(FileNotFoundError)

  create OUTPUT_DIR if needed
  image_extensions ← {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

  # ─────────────────────────────────────────────────────────────
  # 3. DISCOVER AND VALIDATE CALIBRATION IMAGES
  # ─────────────────────────────────────────────────────────────
  paths ← sorted supported files in DATA_DIR

  IF paths is empty:
      RETURN FAIL(FileNotFoundError)

  readable_views ← []

  FOR path IN paths:
      image ← read_image(path)

      IF image is unreadable:
          LOG "[SKIPPED] <name>: image could not be read"
          CONTINUE

      readable_views.append({
          name  : path.name,
          path  : path,
          image : image
      })

  IF readable_views is empty:
      RETURN FAIL(RuntimeError)

  # ─────────────────────────────────────────────────────────────
  # 4. DETECT AND REFINE CHESSBOARD CORNERS
  # ─────────────────────────────────────────────────────────────
  views ← []

  FOR item IN readable_views:
      gray ← convert_to_grayscale(item.image)
      found, corners ← findChessboardCorners(gray, PATTERN)

      IF NOT found OR corners is NULL:
          LOG "[SKIPPED] <name>: chessboard not detected"
          CONTINUE

      refined ← cornerSubPix(
          gray,
          corners,
          window = (11, 11),
          zero_zone = (-1, -1),
          criteria = (EPS + MAX_ITER, 30, 1e-3)
      )

      image_points ← reshape(refined, (N_POINTS, 2))

      views.append({
          name   : item.name,
          path   : item.path,
          image  : item.image,
          im_pts : image_points
      })

  IF number_of(views) < MIN_VALID_VIEWS:
      RETURN FAIL(RuntimeError)

  # ─────────────────────────────────────────────────────────────
  # 5. BUILD METRIC PLANAR COORDINATES
  # ─────────────────────────────────────────────────────────────
  plane_points ← zeros((N_POINTS, 2))
  k ← 0

  FOR y IN 0 .. PATTERN.height - 1:
      FOR x IN 0 .. PATTERN.width - 1:
          plane_points[k] ← (
              x * SQUARE_SIZE_M,
              y * SQUARE_SIZE_M
          )
          k ← k + 1

  FOR view IN views:
      view.plane_pts ← copy(plane_points)

  # ─────────────────────────────────────────────────────────────
  # 6. DEFINE HOMOGENEOUS-COORDINATE CONVERSION
  # ─────────────────────────────────────────────────────────────
  FUNCTION to_homogeneous(points):
      RETURN column_stack(points, ones(number_of(points)))

  # ─────────────────────────────────────────────────────────────
  # 7. NORMALIZE PLANE AND IMAGE POINTS
  # ─────────────────────────────────────────────────────────────
  FUNCTION normalization_transform(points):
      centroid ← mean(points)
      shifted ← points - centroid
      mean_distance ← mean(euclidean_norm(shifted))

      IF mean_distance < 1e-12:
          RETURN FAIL(ValueError)

      scale ← sqrt(2) / mean_distance

      RETURN [
          [scale, 0,     -scale * centroid.x],
          [0,     scale, -scale * centroid.y],
          [0,     0,      1]
      ]

  FOR view IN views:
      view.T_image ← normalization_transform(view.im_pts)
      view.T_plane ← normalization_transform(view.plane_pts)

      view.normalized_image ←
          (view.T_image @ to_homogeneous(view.im_pts).T).T

      view.normalized_plane ←
          (view.T_plane @ to_homogeneous(view.plane_pts).T).T

  # ─────────────────────────────────────────────────────────────
  # 8. BUILD THE NORMALIZED DLT SYSTEM
  # ─────────────────────────────────────────────────────────────
  FUNCTION build_DLT_matrix(norm_plane, norm_image):
      Q ← zeros((2 * N_POINTS, 9))

      FOR i IN 0 .. N_POINTS - 1:
          X, Y, _ ← norm_plane[i]
          u, v, _ ← norm_image[i]

          Q[2*i] ←
              [X, Y, 1, 0, 0, 0, -u*X, -u*Y, -u]

          Q[2*i + 1] ←
              [0, 0, 0, X, Y, 1, -v*X, -v*Y, -v]

      RETURN Q

  FOR view IN views:
      view.Q ← build_DLT_matrix(
          view.normalized_plane,
          view.normalized_image
      )

      U, singular_values, Vt ← svd(view.Q)
      h ← Vt[-1]

      view.H_normalized ← reshape(h, (3, 3))
      view.dlt_singular_values ← singular_values

  # ─────────────────────────────────────────────────────────────
  # 9. DENORMALIZE EVERY HOMOGRAPHY
  # ─────────────────────────────────────────────────────────────
  FOR view IN views:
      H ← inv(view.T_image) @ view.H_normalized @ view.T_plane

      IF abs(H[2, 2]) < 1e-12:
          RETURN FAIL(ValueError)

      H ← H / H[2, 2]

      IF H is not finite:
          RETURN FAIL(ValueError)

      IF shape(view.im_pts) != (N_POINTS, 2):
          RETURN FAIL(ValueError)

      IF shape(view.plane_pts) != (N_POINTS, 2):
          RETURN FAIL(ValueError)

      view.H ← H

  # ─────────────────────────────────────────────────────────────
  # 10. BUILD ZHANG MULTI-VIEW CONSTRAINTS
  # ─────────────────────────────────────────────────────────────
  FUNCTION v_ij(hi, hj):
      RETURN [
          hi[0]*hj[0],
          hi[0]*hj[1] + hi[1]*hj[0],
          hi[1]*hj[1],
          hi[2]*hj[0] + hi[0]*hj[2],
          hi[2]*hj[1] + hi[1]*hj[2],
          hi[2]*hj[2]
      ]

  rows ← []

  FOR view IN views:
      h1 ← view.H[:, 0]
      h2 ← view.H[:, 1]

      rows.append(v_ij(h1, h2))
      rows.append(v_ij(h1, h1) - v_ij(h2, h2))

  V ← stack(rows)
  Uv, singular_values_V, VtV ← svd(V)
  b ← VtV[-1]

  # ─────────────────────────────────────────────────────────────
  # 11. RECOVER THE INTRINSIC MATRIX K
  # ─────────────────────────────────────────────────────────────
  b11, b12, b22, b13, b23, b33 ← b

  denominator ← b11*b22 - b12^2

  IF abs(denominator) < 1e-12:
      RETURN FAIL(ValueError)

  v0 ← (b12*b13 - b11*b23) / denominator

  lambda ←
      b33
      - (b13^2 + v0*(b12*b13 - b11*b23)) / b11

  IF lambda/b11 <= 0 OR lambda*b11/denominator <= 0:
      b ← -b
      recompute b11, b12, b22, b13, b23, b33
      recompute denominator, v0, lambda

  alpha ← sqrt(lambda / b11)
  beta  ← sqrt(lambda * b11 / denominator)
  gamma ← -b12 * alpha^2 * beta / lambda
  u0    ← gamma*v0/beta - b13*alpha^2/lambda

  K ← [
      [alpha, gamma, u0],
      [0,     beta,  v0],
      [0,     0,     1]
  ]

  # ─────────────────────────────────────────────────────────────
  # 12. RECOVER ONE CAMERA POSE PER VIEW
  # ─────────────────────────────────────────────────────────────
  FUNCTION recover_pose(K, H):
      Kinv ← inv(K)

      h1 ← H[:, 0]
      h2 ← H[:, 1]
      h3 ← H[:, 2]

      lambda_pose ← 1 / norm(Kinv @ h1)

      r1 ← lambda_pose * (Kinv @ h1)
      r2 ← lambda_pose * (Kinv @ h2)
      r3 ← cross(r1, r2)

      R_approx ← column_stack(r1, r2, r3)

      U, _, Vt ← svd(R_approx)
      R ← U @ Vt

      IF det(R) < 0:
          flip the final column of U
          R ← U @ Vt

      t ← lambda_pose * (Kinv @ h3)

      RETURN R, t

  FOR view IN views:
      view.R, view.t ← recover_pose(K, view.H)

  # ─────────────────────────────────────────────────────────────
  # 13. REPROJECT CALIBRATION POINTS
  # ─────────────────────────────────────────────────────────────
  FOR view IN views:
      world_xyz ← column_stack(
          view.plane_pts,
          zeros(N_POINTS)
      )

      camera_points ← world_xyz @ view.R.T + view.t
      projected_h ← camera_points @ K.T

      IF any(abs(projected_h[:, 2]) < 1e-12):
          RETURN FAIL(ValueError)

      view.world_xyz ← world_xyz
      view.projected ←
          projected_h[:, :2] / projected_h[:, 2:3]

  # ─────────────────────────────────────────────────────────────
  # 14. COMPUTE CALIBRATION ERRORS
  # ─────────────────────────────────────────────────────────────
  all_errors ← []

  FOR view IN views:
      view.residuals ← view.projected - view.im_pts
      view.errors ← euclidean_norm(view.residuals, axis = 1)

      view.mean_error ← mean(view.errors)
      view.rmse ← sqrt(mean(view.errors^2))

      all_errors.append(view.errors)

  all_errors ← concatenate(all_errors)

  overall_mean_error ← mean(all_errors)
  overall_rmse ← sqrt(mean(all_errors^2))

  # ─────────────────────────────────────────────────────────────
  # 15. GENERATE THE REQUIRED DIAGNOSTICS
  # ─────────────────────────────────────────────────────────────
  save "detected_chessboard_corners.png"
  save "homography_estimation_pipeline.png"
  save "estimated_camera_poses.png"
  save "reprojection_results.png"
  save "mean_reprojection_error_by_view.png"
  save "reprojection_error_distribution.png"

  # ─────────────────────────────────────────────────────────────
  # 16. RUN FINAL VALIDATION
  # ─────────────────────────────────────────────────────────────
  CHECK shape(K) == (3, 3)
  CHECK K is finite

  FOR view IN views:
      CHECK view.H is finite
      CHECK view.R.T @ view.R is close to identity
      CHECK det(view.R) is close to +1
      CHECK view.errors are finite

  CHECK all six required diagnostic files exist

  IF any implemented check fails:
      RETURN FAIL(RuntimeError)

  PRINT "All Camera Calibration validation checks passed."
  RETURN SUCCESS(K, views, overall_mean_error, overall_rmse)

END ALGORITHM
```

---

# Part II — Step-by-step elaboration

Each step below is the six-section contract corresponding to a numbered section in Part I:

- **Purpose** — why the step exists.
- **Inputs** — artifacts that must already exist.
- **Procedure** — the ordered computation.
- **Produces** — outputs, shapes, or invariants created by the step.
- **Acceptance contract** — conditions that must hold before proceeding.
- **Failure mode** — what failure means and how execution should respond.

**Standing rule.** The acceptance contract of step *k* is the precondition of step *k+1*.

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Open the lab with the project environment selected in VS Code.
2. Treat this notebook as the coding plan for `main.ipynb`; do not compute results here.
3. Proceed only in the order below because later calibration quantities depend on earlier validated geometry.

**Produces**

Selected environment and a top-to-bottom execution plan.

**Acceptance contract**

Review the local assumptions before using cached notebook outputs.

**Failure mode**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create the Setup

**Purpose**

Fix the calibration geometry and numerical environment.

**Inputs**

Python environment; 8 by 6 internal corners.

**Procedure**

1. Import `Path`, OpenCV, NumPy, and Matplotlib.
2. Set NumPy display precision so matrices and residuals are readable.
3. Define square size, 8×6 internal corners, minimum valid views, and corner-refinement criteria.

**Produces**

Imports; 0.03 m spacing; minimum 3 views.

**Acceptance contract**

Check the board uses internal corners, not numbers of squares.

**Failure mode**

Stop on missing dependencies or an incorrect board specification.

## 2. Create Repository-Relative Paths

**Purpose**

Keep the experiment portable within the lab.

**Inputs**

Lab notebooks directory.

**Procedure**

1. Create the calibration-image path and `outputs/figures/` path.
2. Create the output directory if needed.
3. Define supported image extensions; use no machine-specific absolute path.

**Produces**

DATA_DIR and OUTPUT_DIR paths.

**Acceptance contract**

Input directory exists; output directory is writable.

**Failure mode**

Resolve the working directory before loading data.

## 3. Discover and Validate Calibration Images

**Purpose**

Build a deterministic image inventory.

**Inputs**

Supported files under data/calibration_images.

**Procedure**

1. Enumerate supported image files and sort them deterministically.
2. If no image is found, stop with `FileNotFoundError`.
3. Create containers for readable images and retained calibration views.

**Produces**

Sorted paths and readable_calibration_views.

**Acceptance contract**

At least one readable image; retain names alongside arrays.

**Failure mode**

Log and skip unreadable files; stop if none remain.

## 4. Detect and Refine Chessboard Corners

**Purpose**

Obtain consistent subpixel correspondences.

**Inputs**

Readable BGR images; REFINE_CRITERIA.

**Procedure**

1. For each image: read it and convert it to grayscale.
2. Detect the complete 8×6 chessboard pattern.
3. If found, refine corners with `cornerSubPix`; otherwise reject that view.
4. Store image name, image array, and refined 2-D points.
5. Stop if fewer than the required number of valid views remain.

**Produces**

views with im_pts shaped (48, 2), in pixels.

**Acceptance contract**

Complete 8 by 6 pattern; at least 3 retained views.

**Failure mode**

Skip incomplete detections explicitly; stop below MIN_VALID_VIEWS.

## 5. Build Metric Planar Coordinates

**Purpose**

Attach metric board coordinates to detected corners.

**Inputs**

Board dimensions and SQUARE_SIZE_M.

**Procedure**

1. Create 48 planar points in the same ordering as detected corners.
2. Set each point to `(x·square_size, y·square_size)` in metres.
3. Keep the board on `Z=0`; add the zero z-coordinate only when 3-D points are needed.

**Produces**

plane_pts shaped (48, 2), in metres.

**Acceptance contract**

x varies fastest, matching corner ordering; board lies on Z=0.

**Failure mode**

Resolve ordering or units before estimating homographies.

## 6. Define Homogeneous-Coordinate Conversion

**Purpose**

Enable homogeneous matrix multiplication.

**Inputs**

Plane/image points shaped (N, 2).

**Procedure**

1. Create a helper that appends a homogeneous coordinate of 1 to every 2-D point.
2. Use it for both plane and image correspondences.
3. Validate the returned shape before matrix computations.

**Produces**

Homogeneous arrays shaped (N, 3).

**Acceptance contract**

Last column equals 1; first two columns preserve coordinates.

**Failure mode**

Do not pass arrays with missing or extra coordinate columns.

## 7. Normalize Plane and Image Points

**Purpose**

Balance coordinate magnitudes before solving DLT.

**Inputs**

48 plane and image points per view.

**Procedure**

1. Compute each point-set centroid and shift to zero mean.
2. Compute mean Euclidean distance from the centroid.
3. Set scale to `sqrt(2)/mean_distance`; reject coincident-point configurations.
4. Build each 3×3 normalization transform and apply it.
5. Store `T_plane`, `T_image`, and normalized point sets per view.

**Produces**

T_plane, T_image and normalized points.

**Acceptance contract**

Centroid approximately zero; mean radius approximately sqrt(2). Current coincidence guard: mean distance below 1e-12.

**Failure mode**

Stop on coincident points; a large normalization scale alone is not degeneracy.

## 8. Build the Normalized DLT System

**Purpose**

Estimate a planar mapping up to scale.

**Inputs**

Normalized correspondences.

**Procedure**

1. For every correspondence add the two standard DLT rows to matrix `Q`.
2. Solve `Qh=0` by SVD.
3. Select the right-singular vector for the smallest singular value.
4. Reshape it into a normalized 3×3 homography.

**Produces**

Q shaped (96, 9); singular values; H_normalized shaped (3, 3).

**Acceptance contract**

Finite Q and homography; use the last right-singular vector. An exact solution is expected to have a near-zero last singular value.

**Failure mode**

Inspect degenerate geometry if the solution is not isolated; do not reject an exact fit because sigma_max/sigma_min is large.

## 9. Denormalize Every Homography

**Purpose**

Express the homography in metres-to-pixels coordinates.

**Inputs**

H_normalized and both normalization matrices.

**Procedure**

1. Compute `H = inv(T_image) @ H_normalized @ T_plane`.
2. Normalize by `H[2,2]`; reject numerically degenerate scale.
3. Store one finite 3×3 homography per retained view.
4. Recheck that each view still contains 48 plane/image correspondences.

**Produces**

One H shaped (3, 3) per view.

**Acceptance contract**

Current helper rejects abs(H[2,2]) below 1e-12 before scaling; verify 48 paired points.

**Failure mode**

Stop on the helper's scale error. This is a chosen normalization limitation, not proof the plane is parallel to the image.

## 10. Build Zhang Multi-View Constraints

**Purpose**

Combine orthogonality constraints from multiple views.

**Inputs**

Retained homographies.

**Procedure**

1. Extract `h1` and `h2` from every homography.
2. Implement the Zhang `v_ij` vector.
3. Stack `v12` and `v11-v22` for all views into matrix `V`.
4. Solve `Vb=0` by SVD and retain the last right-singular vector `b`.

**Produces**

V shaped (2*number_of_views, 6); unit-norm b shaped (6,).

**Acceptance contract**

Inspect the singular spectrum and residual in the same scaling as V. Current implementation stacks unnormalized rows.

**Failure mode**

Do not infer failure from an absolute residual alone or require noisy data to reach machine precision.

## 11. Recover the Intrinsic Matrix

**Purpose**

Recover the camera's intrinsic parameters.

**Inputs**

b in order (B11,B12,B22,B13,B23,B33).

**Procedure**

1. Unpack `b` into the six symmetric-conic coefficients in the order used by `build_zhang_constraint_vector`.
2. Compute `denominator = b11*b22-b12**2`; the current code raises when its absolute value is below `1e-12`.
3. Compute `v0` and `lambda_`, then the two focal-length radicands.
4. Describe the existing sign-retry branch accurately: it negates `b` and recomputes the expressions, but the radicands are mathematically invariant to a global sign flip.
5. Assemble `K` with the closed-form formulas. Review finiteness and positive focal lengths before interpreting the result; the limitations section distinguishes this review from the current automatic checks.

**Produces**

alpha, beta, gamma, u0, v0 and K shaped (3, 3).

**Acceptance contract**

Review finite radicands and positive focal lengths. Existing final check verifies only K shape and finiteness.

**Failure mode**

Stop analysis if K is nonphysical; sign reversal cannot repair negative radicands, whose ratios are sign-invariant. See implementation limits below.

## 12. Recover One Camera Pose per View

**Purpose**

Recover board-to-camera orientation and translation.

**Inputs**

K and each H.

**Procedure**

1. Compute `K^-1 h1`, `K^-1 h2`, and `K^-1 h3`.
2. Use the norm of `K^-1 h1` for the common pose scale.
3. Form `r1`, `r2`, `r3=r1×r2`, and `t`.
4. Project the approximate rotation to the nearest orthonormal matrix with SVD.
5. If `det(R)<0`, correct the sign so `R` is a proper rotation.
6. Store finite `R` and `t`.

**Produces**

R shaped (3, 3); t shaped (3,), in metres.

**Acceptance contract**

Current scale is 1/norm(K^-1 h1); SVD projection enforces a proper rotation. Check orthogonality and determinant near +1.

**Failure mode**

Do not describe t_z>0 as enforced: current code has no cheirality correction.

## 13. Reproject Calibration Points

**Purpose**

Predict image coordinates from the recovered camera.

**Inputs**

K, R, t; board coordinates on Z=0.

**Procedure**

1. Convert planar points to `(X,Y,0)`.
2. Transform them into the camera frame with `R` and `t`.
3. Project with `K` and divide by homogeneous depth.
4. Reject near-zero depth and store the projected 2-D points.

**Produces**

world_xyz shaped (48, 3); projected shaped (48, 2).

**Acceptance contract**

Current guard checks absolute homogeneous depth against 1e-12. Review positive camera-frame depth separately.

**Failure mode**

Stop on near-zero depth; flag negative depths as physically invalid even if division is finite.

## 14. Compute Calibration Errors

**Purpose**

Measure calibration fit in pixel units.

**Inputs**

Observed and projected points with identical ordering.

**Procedure**

1. Compute residual vectors `projected-observed`.
2. Convert residuals to point-wise Euclidean errors.
3. Compute per-view mean error and RMSE.
4. Concatenate all errors and compute overall mean error and RMSE.

**Produces**

Per-point errors; per-view and global mean/RMSE.

**Acceptance contract**

Euclidean error per point; RMSE=sqrt(mean(error**2)); global metrics pool all points.

**Failure mode**

Investigate nonfinite values or systematic residuals; no universal pixel acceptance threshold is implemented.

## 15. Generate the Required Diagnostics

**Purpose**

Expose detection, geometry and residual quality visually.

**Inputs**

Stored views, K, poses and error arrays.

**Procedure**

1. Save the homography-estimation pipeline figure.
2. Save detected corners, camera poses, and reprojection overlays.
3. Save mean reprojection error by view and the global error distribution.
4. Build every plot from the same validated arrays used for numerical metrics.

**Produces**

Six PNG diagnostics listed in the output contract.

**Acceptance contract**

Overlays and plots must use the same retained views and errors; camera centre is -R.T@t.

**Failure mode**

Regenerate missing plots; do not treat visually plausible overlays as physical validation.

## 16. Run Final Validation

**Purpose**

Check the implemented numerical and artifact conditions.

**Inputs**

Computed calibration and REQUIRED_OUTPUTS.

**Procedure**

1. Follow the final validation cell in `main.ipynb`: check finite `K` with shape `(3,3)`.
2. Check finite homographies, rotations with `R.T @ R` close to identity and determinant close to `+1`, and finite error arrays.
3. Check that all six declared figures exist.
4. Interpret the success message as the result of these implemented checks. Positive depths, finite translations and reprojection-quality thresholds are additional review requirements, not checks currently enforced by that cell.

**Produces**

Final validation outcome.

**Acceptance contract**

K finite 3 by 3; H finite; R.T@R near identity and det(R) near 1; finite errors; six files present.

**Failure mode**

Any existing failed check stops execution; separately report unimplemented physical checks.